# Hamiltonian Monte Carlo for Bayesian Logistic Regression

**Dataset:** Breast Cancer Wisconsin (UCI / sklearn) — 569 samples, binary outcome
**Goal:** Bayesian classification with full posterior uncertainty, implemented from scratch

---

## Why Hamiltonian dynamics for statistics?

In classical mechanics, the Hamiltonian governs the evolution of a system:

$$H(q, p) = U(q) + K(p)$$

In Bayesian statistics we want samples from a posterior $\pi(\beta | \mathbf{X}, y)$.
The key insight (Neal 2011) is to interpret the posterior as a physical landscape:

| Physics | Statistics |
|---|---|
| Position $q$ | Model parameters $\beta$ |
| Momentum $p$ | Auxiliary variable (sampled fresh each step) |
| Potential energy $U(q) = -\log \pi(q \mid \text{data})$ | Negative log-posterior |
| Kinetic energy $K(p) = \frac{1}{2} p^\top M^{-1} p$ | Gaussian momentum |

Hamilton's equations then describe a trajectory that **stays on level sets of $H$**,
allowing the sampler to explore the posterior efficiently without the diffusive
random-walk behavior of simpler MCMC methods.

**Liouville's theorem** (conservation of phase-space volume) is what makes the
Metropolis correction exact: the leapfrog integrator is symplectic, so the
acceptance probability has the right stationary distribution.


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Ellipse

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.calibration import calibration_curve
from sklearn.feature_selection import mutual_info_classif

from src import hmc_sample, potential_energy, grad_potential, kinetic_energy, hamiltonian
from src.diagnostics import effective_sample_size, r_hat, summary
from src.integrator import leapfrog

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 12})


## 2. Data: Breast Cancer Wisconsin

The dataset contains 30 numeric features computed from a digitised image of a fine needle
aspirate of a breast mass.  The outcome is binary: **1 = benign, 0 = malignant**.

We select the **top 5 features by mutual information** to keep visualisation tractable
and computation fast.  The HMC code works identically with all 30 features.


In [ ]:
data = load_breast_cancer()
X_full, y = data.data, data.target

# Select top-5 features by mutual information with the label
mi = mutual_info_classif(X_full, y, random_state=42)
top_idx = np.argsort(mi)[::-1][:5]
feature_names = list(data.feature_names[top_idx])
print("Selected features:")
for i, (idx, name) in enumerate(zip(top_idx, feature_names)):
    print(f"  {i+1}. {name}  (MI = {mi[idx]:.4f})")

X_sel = X_full[:, top_idx]

# Standardise to zero mean, unit variance (important for HMC step-size tuning)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)

# Prepend intercept column
X = np.column_stack([np.ones(len(X_scaled)), X_scaled])
param_names = ['intercept'] + feature_names

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining : {len(X_train)} samples")
print(f"Test     : {len(X_test)} samples")
print(f"Params   : {X_train.shape[1]}  (5 features + intercept)")
print(f"Prevalence (train): {y_train.mean():.1%} benign")


In [ ]:
# ── EDA: feature distributions by class ──────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
colors = {0: '#e05c5c', 1: '#5c8ee0'}
labels = {0: 'Malignant', 1: 'Benign'}

for ax, feat, name in zip(axes, X_scaled.T, feature_names):
    for cls in [0, 1]:
        ax.hist(feat[y == cls], bins=28, alpha=0.6, color=colors[cls],
                label=labels[cls], density=True, edgecolor='white', linewidth=0.3)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Standardised value')

axes[0].set_ylabel('Density')
axes[-1].legend(fontsize=9)
fig.suptitle('Feature Distributions by Class', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 3. The Statistical Hamiltonian

For Bayesian logistic regression the posterior is:

$$
\pi(\beta | \mathbf{X}, y) \propto \underbrace{\prod_{i=1}^n \sigma(x_i^\top \beta)^{y_i}\,[1-\sigma(x_i^\top \beta)]^{1-y_i}}_{\text{likelihood}} \cdot \underbrace{\mathcal{N}(\beta; 0, \sigma_{\text{prior}}^2 I)}_{\text{prior}}
$$

The **potential energy** is the negative log-posterior:

$$
U(\beta) = -\sum_{i=1}^n \bigl[y_i \log \sigma(x_i^\top \beta) + (1-y_i)\log(1-\sigma(x_i^\top \beta))\bigr] + \frac{\|\beta\|^2}{2\sigma_{\text{prior}}^2}
$$

Its gradient (required by the leapfrog integrator) has a clean closed form:

$$
\nabla_\beta U = \mathbf{X}^\top (\sigma(\mathbf{X}\beta) - y) + \frac{\beta}{\sigma_{\text{prior}}^2}
$$


In [ ]:
# ── Numerical gradient check ──────────────────────────────────────────────────
# Verify our analytic gradient against finite differences

beta_test = np.random.randn(X_train.shape[1])
eps_fd = 1e-5
grad_analytic = grad_potential(beta_test, X_train, y_train, sigma_prior=1.0)
grad_fd = np.zeros_like(beta_test)
for j in range(len(beta_test)):
    e = np.zeros_like(beta_test); e[j] = eps_fd
    grad_fd[j] = (potential_energy(beta_test + e, X_train, y_train) -
                  potential_energy(beta_test - e, X_train, y_train)) / (2 * eps_fd)

max_err = np.max(np.abs(grad_analytic - grad_fd))
rel_err = max_err / (np.linalg.norm(grad_fd) + 1e-12)
print(f"Gradient check — max abs error : {max_err:.2e}")
print(f"                 relative error: {rel_err:.2e}  {'✓ PASS' if rel_err < 1e-5 else '✗ FAIL'}")


## 4. Phase Space Warm-Up: Leapfrog on a 2D Gaussian

Before fitting the full model, let's visualise what the leapfrog integrator
actually does in phase space using a 2D Gaussian posterior (analytically tractable).

$$
\pi(q) = \mathcal{N}(0, \Sigma), \quad
U(q) = \tfrac{1}{2}\,q^\top \Sigma^{-1} q, \quad
\nabla U(q) = \Sigma^{-1} q
$$

The leapfrog trajectory should approximately follow a **constant-$H$ contour** (an
ellipse in phase space).  Any deviation from the exact ellipse is the numerical
integration error that the Metropolis correction absorbs.


In [ ]:
np.random.seed(7)

# Define a correlated 2D Gaussian posterior
Sigma = np.array([[2.0, 1.2], [1.2, 1.0]])
Sigma_inv = np.linalg.inv(Sigma)
grad_U_toy = lambda q: Sigma_inv @ q
U_toy      = lambda q: 0.5 * q @ Sigma_inv @ q

# Initial state
q0 = np.array([2.0, -1.5])
p0 = np.array([-0.5,  1.8])

# Run leapfrog with two different step sizes
eps_fine   = 0.15
eps_coarse = 0.60
L_steps    = 30

_, _, q_traj_fine,   p_traj_fine   = leapfrog(q0, p0, grad_U_toy, eps_fine,   L_steps)
_, _, q_traj_coarse, p_traj_coarse = leapfrog(q0, p0, grad_U_toy, eps_coarse, L_steps)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, q_traj, p_traj, eps, label, colour in [
    (axes[0], q_traj_fine,   p_traj_fine,   eps_fine,   f'ε={eps_fine}',   '#3a7abf'),
    (axes[1], q_traj_coarse, p_traj_coarse, eps_coarse, f'ε={eps_coarse}', '#bf3a3a'),
]:
    # True constant-H ellipse
    H0 = U_toy(q0) + 0.5 * np.dot(p0, p0)
    theta = np.linspace(0, 2 * np.pi, 400)
    # approximate as circle for display; real ellipse omitted for clarity

    ax.plot(q_traj[:, 0], p_traj[:, 1], 'o-', lw=1.8, ms=5, color=colour, alpha=0.85)
    ax.plot(q_traj[0, 0],  p_traj[0, 1],  'go', ms=10, zorder=5, label='Start')
    ax.plot(q_traj[-1, 0], p_traj[-1, 1], 'r*', ms=13, zorder=5, label='End')
    ax.set_title(f'Phase Space Trajectory  ({label},  L={L_steps})', fontweight='bold')
    ax.set_xlabel('Position  $q_1$  (parameter)')
    ax.set_ylabel('Momentum  $p_2$')
    ax.legend(fontsize=9)

    # Hamiltonian values along trajectory
    H_vals = np.array([U_toy(q) + 0.5 * np.dot(p, p)
                       for q, p in zip(q_traj, p_traj)])
    print(f"ε={eps:.2f} | H range: [{H_vals.min():.4f}, {H_vals.max():.4f}]"
          f"  ΔH/H₀ = {(H_vals.max()-H_vals.min())/H_vals[0]:.2e}")

plt.suptitle('Leapfrog Symplectic Integrator — Phase Space', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Hamiltonian conservation plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

for ax, q_traj, p_traj, eps, colour in [
    (axes[0], q_traj_fine,   p_traj_fine,   eps_fine,   '#3a7abf'),
    (axes[1], q_traj_coarse, p_traj_coarse, eps_coarse, '#bf3a3a'),
]:
    H_vals = np.array([U_toy(q) + 0.5 * np.dot(p, p)
                       for q, p in zip(q_traj, p_traj)])
    ax.plot(H_vals, 'o-', lw=1.5, ms=4, color=colour, alpha=0.85)
    ax.axhline(H_vals[0], ls='--', lw=1.2, color='k', alpha=0.5, label=f'H₀ = {H_vals[0]:.4f}')
    ax.set_title(f'H Conservation  (ε={eps})', fontweight='bold')
    ax.set_xlabel('Leapfrog step')
    ax.set_ylabel('H(q, p) = U + K')
    ax.legend(fontsize=9)

fig.suptitle('Leapfrog Preserves H up to O(ε²) — Metropolis Corrects the Rest',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 5. Running HMC on Breast Cancer

### Hyperparameter rationale

| Parameter | Value | Why |
|---|---|---|
| `epsilon` | 0.08 | Small enough for ~85 % acceptance after standardisation |
| `L` | 30 | Long trajectory = large moves, low autocorrelation |
| `n_warmup` | 600 | Discard transient from initialisation |
| `n_samples` | 2 000 | Enough for stable ESS estimates |
| `sigma_prior` | 2.0 | Weakly informative on standardised scale |

A **well-tuned HMC** should show 60–90 % acceptance rate and ESS > 100 per parameter.


In [ ]:
d = X_train.shape[1]
q_init = np.zeros(d)          # start at prior mean

print("Running HMC chain …")
samples, info = hmc_sample(
    q_init     = q_init,
    X          = X_train,
    y          = y_train,
    n_samples  = 2000,
    epsilon    = 0.08,
    L          = 30,
    sigma_prior= 2.0,
    n_warmup   = 600,
    seed       = 42,
    verbose    = True,
)

last_q_traj = info['last_q_traj']
last_p_traj = info['last_p_traj']
print(f"\nSamples shape: {samples.shape}  (draws × params)")


## 6. MCMC Diagnostics

Good MCMC diagnostics require:

- **Trace plots** — should look like "hairy caterpillars" (well-mixed)
- **Effective Sample Size (ESS)** — target ≥ 100 per parameter
- **R̂ (Gelman-Rubin)** — near 1.0 indicates convergence (< 1.05 is good)
- **Energy conservation** — confirms the leapfrog step size is appropriate


In [ ]:
# ── Trace plots + marginal posteriors ────────────────────────────────────────
fig, axes = plt.subplots(d, 2, figsize=(13, 2.5 * d))
fig.suptitle('MCMC Diagnostics — Trace Plots & Marginal Posteriors', fontsize=13,
             fontweight='bold', y=1.001)

for i, name in enumerate(param_names):
    chain = samples[:, i]
    ax_t, ax_h = axes[i]

    # Trace
    ax_t.plot(chain, lw=0.5, alpha=0.8, color='#3a7abf')
    ax_t.axhline(chain.mean(), color='red', lw=1.2, ls='--', alpha=0.7)
    ax_t.set_ylabel(name, fontsize=9)
    ax_t.set_xlabel('Iteration' if i == d - 1 else '')
    if i == 0:
        ax_t.set_title('Trace', fontsize=10)

    # Histogram
    ax_h.hist(chain, bins=45, density=True, color='#3a7abf', alpha=0.75,
              edgecolor='white', lw=0.25)
    ax_h.axvline(chain.mean(), color='red', lw=1.2, ls='--',
                 label=f'μ={chain.mean():.3f}')
    q025, q975 = np.percentile(chain, [2.5, 97.5])
    ax_h.axvspan(q025, q975, alpha=0.12, color='red', label='95 % CI')
    ax_h.set_xlabel(name, fontsize=9)
    if i == 0:
        ax_h.set_title('Posterior', fontsize=10)
    ax_h.legend(fontsize=7.5)

plt.tight_layout()
plt.show()


In [ ]:
# ── ESS and summary statistics ────────────────────────────────────────────────
rows = summary(samples, param_names)
print(f"{'Param':<28} {'Mean':>8} {'Std':>7} {'2.5%':>8} {'97.5%':>8} {'ESS':>7}")
print('─' * 70)
for r in rows:
    print(f"{r['param']:<28} {r['mean']:>8.4f} {r['std']:>7.4f} "
          f"{r['2.5%']:>8.4f} {r['97.5%']:>8.4f} {r['ESS']:>7.0f}")

min_ess = min(r['ESS'] for r in rows)
print(f"\nMinimum ESS across parameters: {min_ess:.0f}")
print(f"Acceptance rate : {info['acceptance_rate']:.1%}")


In [ ]:
# ── Energy conservation on the last leapfrog trajectory ───────────────────────
H_vals = np.array([
    potential_energy(q, X_train, y_train, sigma_prior=2.0) + kinetic_energy(p)
    for q, p in zip(last_q_traj, last_p_traj)
])
delta_H = np.abs(H_vals - H_vals[0])

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

axes[0].plot(H_vals, 'o-', lw=1.5, ms=3.5, color='#bf7a3a')
axes[0].axhline(H_vals[0], ls='--', lw=1, color='k', alpha=0.5,
                label=f'H₀ = {H_vals[0]:.2f}')
axes[0].set_title('Hamiltonian Along Last Leapfrog Trajectory', fontweight='bold')
axes[0].set_xlabel('Leapfrog step'); axes[0].set_ylabel('H(q, p)')
axes[0].legend(fontsize=9)

axes[1].semilogy(delta_H + 1e-12, 'o-', lw=1.5, ms=3.5, color='#7a3abf')
axes[1].set_title('|ΔH| = |H(step k) − H₀|  (log scale)', fontweight='bold')
axes[1].set_xlabel('Leapfrog step'); axes[1].set_ylabel('|ΔH|')

print(f"Max |ΔH| = {delta_H.max():.4f}   (should be ≪ 1 for good step size)")
plt.tight_layout()
plt.show()


## 7. Posterior Predictive Distribution

Rather than a single point estimate, Bayesian logistic regression gives a **full
predictive distribution**:

$$
p(y^* = 1 \mid x^*, \mathbf{X}, y) = \int \sigma(x^{*\top} \beta)\, \pi(\beta \mid \mathbf{X}, y)\, d\beta \approx \frac{1}{S} \sum_{s=1}^S \sigma(x^{*\top} \beta^{(s)})
$$

The **predictive uncertainty** (variance over posterior draws) is high for points
near the decision boundary and low for points confidently inside one class.


In [ ]:
def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

# Posterior predictive probabilities on test set
logits_samples = X_test @ samples.T           # (n_test, n_samples)
prob_samples   = sigmoid(logits_samples)       # (n_test, n_samples)
prob_mean      = prob_samples.mean(axis=1)     # predictive mean
prob_std       = prob_samples.std(axis=1)      # predictive std (epistemic uncertainty)

y_pred_hmc = (prob_mean >= 0.5).astype(int)
acc_hmc    = accuracy_score(y_test, y_pred_hmc)
brier_hmc  = brier_score_loss(y_test, prob_mean)
auc_hmc    = roc_auc_score(y_test, prob_mean)

print(f"HMC Bayesian Logistic Regression")
print(f"  Accuracy  : {acc_hmc:.3f}")
print(f"  ROC-AUC   : {auc_hmc:.4f}")
print(f"  Brier score: {brier_hmc:.4f}  (lower is better)")


In [ ]:
# ── Calibration curve ─────────────────────────────────────────────────────────
fraction_pos, mean_pred = calibration_curve(y_test, prob_mean, n_bins=8)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Calibration
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.2, label='Perfect calibration')
axes[0].plot(mean_pred, fraction_pos, 'o-', lw=2, ms=7, color='#3a7abf',
             label='HMC predictive')
axes[0].fill_between(mean_pred, fraction_pos, mean_pred,
                     alpha=0.15, color='#3a7abf')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction of positives')
axes[0].set_title('Calibration Curve', fontweight='bold')
axes[0].legend(fontsize=9)

# Predictive uncertainty
sorted_idx = np.argsort(prob_mean)
ax = axes[1]
ax.scatter(range(len(y_test)), prob_mean[sorted_idx],
           c=y_test[sorted_idx], cmap='RdYlBu', s=18, zorder=3, alpha=0.85)
ax.fill_between(range(len(y_test)),
                (prob_mean - 2*prob_std)[sorted_idx],
                (prob_mean + 2*prob_std)[sorted_idx],
                alpha=0.25, color='grey', label='±2 posterior std')
ax.axhline(0.5, ls='--', lw=1, color='k', alpha=0.5)
ax.set_xlabel('Test sample (sorted by predicted probability)')
ax.set_ylabel('Predicted P(benign)')
ax.set_title('Predictive Mean ± Uncertainty', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Posterior Predictive on Test Set', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 8. Comparison: HMC Posterior vs. Maximum Likelihood Estimate

The MLE point estimate (sklearn `LogisticRegression`) maximises the likelihood
without quantifying uncertainty.  We overlay the MLE on the posterior marginals
to check they are consistent (the MLE should fall near the posterior mode).

A key advantage of HMC is that the **posterior width** gives us calibrated
uncertainty intervals for each coefficient — something the MLE alone cannot provide.


In [ ]:
# Fit sklearn MLE
mle = LogisticRegression(C=4.0, fit_intercept=False, solver='lbfgs', max_iter=1000)
mle.fit(X_train, y_train)
beta_mle = mle.coef_[0]

acc_mle  = accuracy_score(y_test, mle.predict(X_test))
brier_mle = brier_score_loss(y_test, mle.predict_proba(X_test)[:, 1])
auc_mle   = roc_auc_score(y_test, mle.predict_proba(X_test)[:, 1])

print("Comparison:")
print(f"{'':>26} {'Accuracy':>10} {'ROC-AUC':>10} {'Brier':>8}")
print('─' * 56)
print(f"{'HMC Bayesian Logistic':>26} {acc_hmc:>10.3f} {auc_hmc:>10.4f} {brier_hmc:>8.4f}")
print(f"{'MLE (sklearn)':>26} {acc_mle:>10.3f} {auc_mle:>10.4f} {brier_mle:>8.4f}")


In [ ]:
# ── Posterior vs MLE: marginal comparison ─────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
axes = axes.ravel()

for i, (ax, name, b_mle) in enumerate(zip(axes, param_names, beta_mle)):
    chain = samples[:, i]
    q025, q975 = np.percentile(chain, [2.5, 97.5])

    ax.hist(chain, bins=40, density=True, color='#3a7abf', alpha=0.65,
            edgecolor='white', lw=0.25, label='HMC posterior')
    ax.axvline(chain.mean(), color='#3a7abf', lw=2, label=f'Bayes mean {chain.mean():.3f}')
    ax.axvspan(q025, q975, alpha=0.12, color='#3a7abf', label='95 % CI')
    ax.axvline(b_mle, color='#bf3a3a', lw=2, ls='--', label=f'MLE  {b_mle:.3f}')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Coefficient value')
    ax.legend(fontsize=7)

# Hide the unused 6th subplot
if len(param_names) < len(axes):
    axes[-1].set_visible(False)

fig.suptitle('Posterior Marginals vs. MLE Point Estimates', fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 9. Joint Posterior — Pair Plot

The off-diagonal panels reveal **posterior correlations** between parameters — a key
feature invisible to MLE.  Correlated parameters indicate that features carry
redundant information; the posterior captures this joint uncertainty.


In [ ]:
# Thin samples for readability
thin_idx = np.arange(0, len(samples), 4)
S = samples[thin_idx]

fig, axes = plt.subplots(d, d, figsize=(11, 10))
cmap = plt.cm.Blues

for i in range(d):
    for j in range(d):
        ax = axes[i][j]
        if i == j:
            ax.hist(S[:, i], bins=35, density=True,
                    color='#3a7abf', alpha=0.75, edgecolor='white', lw=0.2)
            ax.axvline(S[:, i].mean(), color='red', lw=1.2, ls='--')
        elif i > j:
            ax.scatter(S[:, j], S[:, i], s=1.5, alpha=0.25, color='#3a7abf')
            rho = np.corrcoef(S[:, j], S[:, i])[0, 1]
            ax.set_title(f'ρ={rho:.2f}', fontsize=7, pad=1)
        else:
            ax.set_visible(False)

        if i == d - 1:
            ax.set_xlabel(param_names[j], fontsize=7)
        if j == 0 and i > 0:
            ax.set_ylabel(param_names[i], fontsize=7)
        ax.tick_params(labelsize=6)

fig.suptitle('Joint Posterior — Pair Plot', fontsize=13, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()


## 10. Summary

### What this example demonstrates

| Concept | Where it appears |
|---|---|
| Hamiltonian $H = U + K$ | `src/hamiltonian.py` — exact formula |
| Symplectic integrator | `src/integrator.py` — leapfrog preserves H to O(ε²) |
| Liouville's theorem | Guarantees phase-space volume preserved → Metropolis correction is exact |
| Metropolis acceptance | `src/sampler.py` — absorbs numerical integration error |
| Bayesian inference | Full posterior over $\beta$ instead of a point estimate |
| Uncertainty quantification | 95 % credible intervals + predictive std per test sample |
| Calibration | Posterior predictive probabilities are well-calibrated by construction |

### Take-aways

1. **HMC is not just metaphor** — the physics is exact and guarantees the
   correct stationary distribution via detailed balance.

2. **Posterior correlations matter** — the pair plot reveals structure that
   MLE + standard errors misses, especially when features are correlated.

3. **Small step size ≠ better** — a well-tuned leapfrog with large L explores
   the posterior efficiently; the Metropolis step handles residual error.

4. **ESS / n_samples** is the right efficiency metric — a chain with 10 %
   ESS ratio samples autocorrelated draws, wasting compute.

### Natural extensions

- **NUTS** (No-U-Turn Sampler): automatic L selection, used in Stan and PyMC
- **Mass matrix adaptation**: diagonal M ≈ posterior precision improves mixing
- **Gradient through ODE**: use autograd to do inference on the physical
  Hamiltonian parameters from trajectory data (double Hamiltonian demo)
